# STAGE 1

In this stage I build a tokenizer from scratch for data preparation and sampling purposes.

Here for dataset I am using WikiText-2 from huggingface and TinyStories.

Steps followed usually :     
1. Split the text into individual word and subwords
2. Convert these tokens into token IDs.

In [1]:
! pip install datasets

In [2]:
from datasets import load_dataset

wiki = load_dataset("wikitext", "wikitext-2-raw-v1")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [3]:
wiki_text = wiki['train']['text']

It seems as if theres a lot of empty spaces i should clean them first.

In [4]:
wiki_clean = [item for item in wiki_text if item.strip() != ' ']

In [5]:
ds = load_dataset("roneneldan/TinyStories")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [6]:
print(type(ds))

<class 'datasets.dataset_dict.DatasetDict'>


In [7]:
stories_text = list(ds['train']['text'])

In [8]:
dataset = wiki_clean + stories_text

In [9]:
print(len(dataset))

2156437


In [10]:
dataset = [item for item in dataset if not item.strip().startswith("=")]

In [11]:
print(len(dataset))

2150225


In [12]:
vocab = []

for line in range(749999):
    words = dataset[line].split()
    for word in words:
        vocab.append(" ".join(list(word)) + " </w>")

: 

: 

: 

In [14]:
print(len(vocab))

128938699


In [15]:
from collections import Counter

Cant work with this need to make changes

In [4]:
import time
import heapq

In [16]:
vocab_freq = Counter(vocab)
print(len(vocab_freq))
print(type(vocab_freq))

231399
<class 'collections.Counter'>


In [ ]:
class TrainerV2():
  def __init__(self, vocab_in, iterations):
    self.freq_dict = vocab_in
    self.iters = iterations

    self.pair_freq = {}
    for tokens, freq in self.freq_dict.items():
      for i in range(len(tokens)-1):
        token_pair = (tokens[i], tokens[i+1])
        self.pair_freq[token_pair] = self.pair_freq.get(token_pair, 0) + freq

    self.heap = [(-freq,pair) for pair, freq in self.pair_freq.items()]
    heapq.heapify(self.heap)

  def get_best_pair(self):
    while self.heap:
      neg_freq ,pair = heapq.heappop(self.heap)
      freq = -neg_freq

      if pair in self.pair_freq and self.pair_freq[pair] == freq :
        return pair
    return None

    
  def merger(self):
      current_best_pair = self.get_best_pair()
      replacement = "".join(current_best_pair)
      if current_best_pair == None:
        return
      new_vocab = {}
      
      for (tokens, freq) in self.freq_dict.items():
        new_tokens = []
        i = 0
        while i < len(tokens)-1:
          if (current_best_pair) != (tokens[i], tokens[i+1]):
            new_tokens.append(tokens[i])
            i+=1

          else :
            new_tokens.append(replacement)

            #Remove previous pair
            if i > 0 :
              if (tokens[i-1],tokens[i]) in self.pair_freq:
                self.pair_freq[(tokens[i-1],tokens[i])] -= freq
                if self.pair_freq[(tokens[i-1],tokens[i])] <= 0 :
                  del self.pair_freq[(tokens[i-1],tokens[i])]
            
            #Remove leading pair
            if i+2 < len(tokens):
              if (tokens[i+1],tokens[i+2]) in self.pair_freq:
                self.pair_freq[(tokens[i+1],tokens[i+2])] -= freq
                if self.pair_freq[(tokens[i+1],tokens[i+2])] <= 0 :
                  del self.pair_freq[(tokens[i+1],tokens[i+2])]

            #Managing original pair_freq
            if current_best_pair in self.pair_freq:
              self.pair_freq[current_best_pair]-=freq
              if self.pair_freq[current_best_pair] <= 0 :
                del self.pair_freq[current_best_pair]

            if i>0:
              self.pair_freq[(tokens[i-1],replacement)] = self.pair_freq.get((tokens[i-1],replacement),0) + freq

            if i+2 < len(tokens) :
              self.pair_freq[(replacement,tokens[i+2])] = self.pair_freq.get((replacement,tokens[i+2]),0) + freq

            i+=2
        if i == len(tokens) - 1:
            new_tokens.append(tokens[i])

        new_vocab[tuple(new_tokens)] = new_vocab.get(tuple(new_tokens), 0) + freq

      self.freq_dict = new_vocab
      self.heap = [(-freq, pair) for pair, freq in self.pair_freq.items()]
      heapq.heapify(self.heap)

  def loop(self):
    start = time.perf_counter()
    merge_rules = []

    for i in range(self.iters):
        current_best = self.get_best_pair()

        if current_best is None:
            break

        new_token = "".join(current_best)
        merge_rules.append((current_best, new_token))

        self.merger()

        end = time.perf_counter()

        if (i+1) % 100 == 0:
            print(f"Iteration Number : {i+1}: Merge rules : {merge_rules}")
            print(f"Time used: {end - start:.6f} seconds\n")

    return merge_rules

In [24]:
vocab_freq_dummy_1 = vocab_freq

In [1]:
print(f"Trainer native timing for 100 iteration : 104.5 seconds")

Trainer native timing for 100 iteration : 104.5 seconds


In [2]:
print(f"Trainerv1 benchmark timing for 100 iterations : 53.64 seconds\n")

Trainerv1 benchmark timing for 100 iterations : 53.64 seconds



In [1]:
print(f"Trainerv1 benchmark timing for 9500 iterations : 1 hr 13 min\n")

Trainerv1 benchmark timing for 9500 iterations : 1 hr 13 min



In [ ]:
class 